In [7]:
# import necessary libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

# Machine Learning Forecasting

## Objective

The objective of this notebook is to evaluate whether machine learning models can improve forecasting performance beyond traditional statistical approaches.

Previous analysis demonstrated that demand is primarily driven by strong seasonality and long-term growth. Statistical forecasting methods, particularly Holt-Winters Exponential Smoothing, achieved strong predictive performance and will serve as the benchmark for all machine learning models evaluated in this notebook.

The analysis will focus on:

- Feature engineering
- Lag and rolling-window features
- Temporal train/test validation
- Machine learning forecasting models
- Feature importance analysis
- Benchmark comparison against Holt-Winters

Success will be measured using out-of-sample forecasting accuracy, with particular emphasis on Mean Absolute Percentage Error (MAPE).

## Data Preparation

In [10]:
project_root = Path.cwd().parent
data_path = project_root / "data" / "processed" / "vw_forecasting_base.csv"
columns = [
    "month",
    "product_id",
    "market_id",
    "category",
    "brand",
    "lifecycle_stage",
    "region",
    "lead_time_months",
    "strategic_priority",
    "actual_demand_units",
    "promo_flag",
    "campaign_intensity",
    "demand_signal_quality",
    "available_supply_units",
    "production_capacity_units",
    "supply_constraint_flag",
    "actual_shipped_units",
    "unfulfilled_demand_units",
    "service_level_pct",
    "ending_inventory_units",
    "inventory_risk_status"
]
df = pd.read_csv(
    data_path,
    header=None,
    names=columns,
    sep=";",
    keep_default_na=False
)

In [12]:
# Display 5 first rows of the dataset
df.head()

,month,product_id,market_id,category,brand,lifecycle_stage,region,lead_time_months,strategic_priority,actual_demand_units,...,campaign_intensity,demand_signal_quality,available_supply_units,production_capacity_units,supply_constraint_flag,actual_shipped_units,unfulfilled_demand_units,service_level_pct,ending_inventory_units,inventory_risk_status
0,1/1/2021,PRD001,MKT001,Fragrance,Aurelian,Core,WE,1,3,1454,...,1.0,High,12250,19533,1,950,504,0.6534,1418,Normal
1,1/1/2021,PRD001,MKT002,Fragrance,Aurelian,Core,WE,1,5,1592,...,1.0,High,12250,19533,1,1128,464,0.7085,1172,Normal
2,1/1/2021,PRD001,MKT003,Fragrance,Aurelian,Core,WE,2,4,1536,...,1.0,High,12250,19533,1,1086,450,0.7070,1227,Normal
3,1/1/2021,PRD001,MKT004,Fragrance,Aurelian,Core,NA,2,4,1200,...,1.0,Medium,12250,19533,1,946,254,0.7883,1095,Normal
4,1/1/2021,PRD001,MKT005,Fragrance,Aurelian,Core,NA,3,2,843,...,1.0,High,12250,19533,1,720,123,0.8541,809,Normal


In [14]:
# Display shape of dataset
rows, cols = df.shape
print(f"Rows: {rows} | Columns: {cols}")

Rows: 17280 | Columns: 21


In [16]:
# Show basic summary statistics
df.describe()

,lead_time_months,strategic_priority,actual_demand_units,promo_flag,campaign_intensity,available_supply_units,production_capacity_units,supply_constraint_flag,actual_shipped_units,unfulfilled_demand_units,service_level_pct,ending_inventory_units
count,17280.000000,17280.000000,17280.000000,17280.000000,17280.000000,17280.00000,17280.000000,17280.000000,17280.000000,17280.000000,17280.000000,17280.000000
mean,2.666667,3.333333,1310.963600,0.127083,1.154167,15229.24375,17255.884722,0.430556,1140.506308,170.457292,4.797783,2482.983333
std,1.027432,1.433762,957.199488,0.333076,0.179656,8642.63890,10812.553231,0.495168,746.364198,303.961595,4.537714,2062.686166
min,1.000000,1.000000,0.000000,0.000000,1.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.368400,0.000000
25%,2.000000,2.000000,691.000000,0.000000,1.000000,9498.50000,10305.000000,0.000000,647.000000,0.000000,0.839775,1121.000000
50%,3.000000,4.000000,1108.000000,0.000000,1.100000,14307.00000,15982.500000,0.000000,1006.000000,37.000000,0.962800,1921.000000
75%,3.250000,4.250000,1691.000000,0.000000,1.225000,19475.75000,21796.750000,1.000000,1490.000000,218.000000,10.000000,3251.000000
max,4.000000,5.000000,11134.000000,1.000000,1.450000,74908.00000,111255.000000,1.000000,8264.000000,4159.000000,10.000000,16616.000000


In [18]:
# Convert 'month' column into datetime object
df["month"] = pd.to_datetime(df["month"], format="%d/%m/%Y", dayfirst=True)

In [20]:
# Check if conversion worked
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17280 entries, 0 to 17279
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   month                      17280 non-null  datetime64[ns]
 1   product_id                 17280 non-null  object        
 2   market_id                  17280 non-null  object        
 3   category                   17280 non-null  object        
 4   brand                      17280 non-null  object        
 5   lifecycle_stage            17280 non-null  object        
 6   region                     17280 non-null  object        
 7   lead_time_months           17280 non-null  int64         
 8   strategic_priority         17280 non-null  int64         
 9   actual_demand_units        17280 non-null  int64         
 10  promo_flag                 17280 non-null  int64         
 11  campaign_intensity         17280 non-null  float64       
 12  dema

## Feature Engineering

In [27]:
# Sort by product_id, market_id, month
df = df.sort_values(
    ["product_id", "market_id", "month"]
).reset_index(drop=True)

In [29]:
df.head()

,month,product_id,market_id,category,brand,lifecycle_stage,region,lead_time_months,strategic_priority,actual_demand_units,...,campaign_intensity,demand_signal_quality,available_supply_units,production_capacity_units,supply_constraint_flag,actual_shipped_units,unfulfilled_demand_units,service_level_pct,ending_inventory_units,inventory_risk_status
0,2021-01-01,PRD001,MKT001,Fragrance,Aurelian,Core,WE,1,3,1454,...,1.00,High,12250,19533,1,950,504,0.6534,1418,Normal
1,2021-02-01,PRD001,MKT001,Fragrance,Aurelian,Core,WE,1,3,1518,...,1.10,Medium,18847,17977,0,1352,166,0.8906,1711,Normal
2,2021-03-01,PRD001,MKT001,Fragrance,Aurelian,Core,WE,1,3,1579,...,1.10,Low,17757,19989,1,1393,186,0.8822,1662,Normal
3,2021-04-01,PRD001,MKT001,Fragrance,Aurelian,Core,WE,1,3,1413,...,1.00,Medium,18542,18876,0,1290,123,0.9130,1251,Normal
4,2021-05-01,PRD001,MKT001,Fragrance,Aurelian,Core,WE,1,3,1434,...,1.15,High,19256,20442,1,1360,74,0.9484,1310,Normal
